In [1]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [2]:
# Load Data 
fact = pd.read_csv(r"C:\Users\admin\Desktop\fact_bitcoin_market.csv")
dim_date = pd.read_csv(r"C:\Users\admin\Desktop\dim_date.csv")
dim_timeframe = pd.read_csv(r"C:\Users\admin\Desktop\dim_timeframe.csv")


In [3]:
# Merge Dimensions 
fact = fact.merge(dim_date[["date_key", "year"]], on="date_key", how="left")
fact = fact.merge(dim_timeframe[["timeframe_key", "timeframe"]], on="timeframe_key", how="left")

fact = fact.sort_values(['timeframe', 'date_key']).reset_index(drop=True)

In [4]:
fact.head()

,date_key,timeframe_key,Open,High,Low,Close,Volume,capital_flow,trade_activity,price_change,...,high_volatility_length,compression,overheating,whale_activity,breakout_up,breakout_down,support,resistance,year,timeframe
0,1,1,13774.99,13775.99,13600.00,13698.00,74.453320,1.019043e+06,1312,-76.99,...,2,0,0,0,0,0,98.00,77.99,2018,15m
1,2,1,13698.00,13775.00,13590.00,13644.95,89.776654,1.228408e+06,895,-53.05,...,3,0,0,0,0,0,54.95,130.05,2018,15m
2,3,1,13644.97,13659.97,13555.02,13570.35,43.920484,5.972011e+05,814,-74.62,...,1,0,0,0,0,0,15.33,89.62,2018,15m
3,4,1,13569.98,13665.00,13520.00,13656.23,58.542067,7.948098e+05,919,86.25,...,1,0,0,0,0,0,136.23,8.77,2018,15m
4,5,1,13656.23,13735.24,13610.27,13632.89,58.900513,8.054309e+05,869,-23.34,...,1,0,0,0,0,0,22.62,102.35,2018,15m


In [5]:
# TIME-SERIES FEATURE ENGINEERING

# Create lag features (past 1 and 2 candles) to capture market history per timeframe
for lag in [1, 2]:
    # Momentum and cash flow lags
    fact[f'return_pct_lag_{lag}'] = fact.groupby('timeframe')['return_pct'].shift(lag)
    fact[f'capital_flow_lag_{lag}'] = fact.groupby('timeframe')['capital_flow'].shift(lag)
    fact[f'order_flow_lag_{lag}'] = fact.groupby('timeframe')['order_flow'].shift(lag)
    
    # Volume and whale activity lags
    fact[f'volume_spike_lag_{lag}'] = fact.groupby('timeframe')['volume_spike'].shift(lag)
    fact[f'whale_activity_lag_{lag}'] = fact.groupby('timeframe')['whale_activity'].shift(lag)


In [6]:
# Create rolling features (5-candle moving average) to capture current market trends
fact['rolling_volatility_5'] = fact.groupby('timeframe')['volatility'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean()
)
fact['rolling_buy_ratio_5'] = fact.groupby('timeframe')['buy_ratio'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean()
)
fact['rolling_trade_activity_5'] = fact.groupby('timeframe')['trade_activity'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean()
)

# Clean data by removing the first two rows of each timeframe containing missing values (NaNs)
fact = fact.dropna(subset=['return_pct_lag_2']).reset_index(drop=True)


In [8]:
# List of all features that the model will use for training and prediction
features = [
    # Original baseline features (Current candle data)
    "price_change", "return_pct", "volatility", "Volume", "capital_flow", 
    "trade_activity", "buy_volume", "sell_volume", "buy_ratio", "sell_ratio", 
    "order_flow", "trend_direction", "trend_length", "trend_strength", 
    "support", "resistance", "breakout_up", "breakout_down", "volume_spike", 
    "high_volatility", "compression", "overheating", "whale_activity",
    
    # New historical features (Lagged features)
    "return_pct_lag_1", "return_pct_lag_2",
    "capital_flow_lag_1", "capital_flow_lag_2",
    "order_flow_lag_1", "order_flow_lag_2",
    "volume_spike_lag_1", "volume_spike_lag_2",
    "whale_activity_lag_1", "whale_activity_lag_2",
    
    # New trend features (Rolling average features)
    "rolling_volatility_5", "rolling_buy_ratio_5", "rolling_trade_activity_5"
]

In [12]:
# Store Results for Power BI
predictions_results = []
metrics_results = []
feature_results = []
confusion_results = []

lookahead = 4  # Number of future candles to look ahead for target labeling

# Loop through each timeframe
for tf in ["15m", "1h", "4h", "1d"]:

    print("\n" + "=" * 60)
    print(f" TIMEFRAME: {tf} ".center(60, "="))
    print("=" * 60)

    # Filter current timeframe
    data = fact[fact["timeframe"] == tf].copy()

    # ==========================================
    # Target Labeling
    # ==========================================

    data["max_future_high"] = (
        data["High"]
        .iloc[::-1]
        .rolling(window=lookahead, min_periods=1)
        .max()
        .iloc[::-1]
    )

    data["max_future_high"] = data["max_future_high"].shift(-1)

    data["potential_return"] = (
        data["max_future_high"] - data["Close"]
    ) / data["Close"]

    data = data.dropna(subset=["potential_return"]).reset_index(drop=True)

    # Make sure data is sorted by time
    data = data.sort_values("date_key").reset_index(drop=True)

    # Split 80% Train / 20% Test
    split_index = int(len(data) * 0.8)

    train = data.iloc[:split_index].copy()
    test = data.iloc[split_index:].copy()

    if train.empty or test.empty:
        print(f"Skipping {tf}: Missing train or test split.")
        continue

    

    # ==========================================
    # Dynamic Threshold
    # ==========================================

    threshold = train["potential_return"].quantile(0.70)

    train["target"] = (
        train["potential_return"] >= threshold
    ).astype(int)

    test["target"] = (
        test["potential_return"] >= threshold
    ).astype(int)

    # ==========================================
    # Features
    # ==========================================

    X_train = train[features]
    y_train = train["target"]

    X_test = test[features]
    y_test = test["target"]

    # ==========================================
    # Train Model
    # ==========================================

    model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        max_depth=10,
        class_weight="balanced"
    )

    model.fit(X_train, y_train)

    # ==========================================
    # Prediction
    # ==========================================

    prediction = model.predict(X_test)

    accuracy = accuracy_score(y_test, prediction)
    precision = precision_score(y_test, prediction)
    recall = recall_score(y_test, prediction)
    f1 = f1_score(y_test, prediction)

    print(f"\n[Metrics for {tf}]")
    print("Accuracy:", accuracy)
    print(classification_report(y_test, prediction))

    metrics_results.append({
        "Timeframe": tf,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

    # ==========================================
    # Confusion Matrix
    # ==========================================

    cm = confusion_matrix(y_test, prediction)

    print("\nConfusion Matrix")
    print(cm)

    confusion_results.append({
        "Timeframe": tf,
        "Actual_0_Pred_0": cm[0, 0],
        "Actual_0_Pred_1": cm[0, 1],
        "Actual_1_Pred_0": cm[1, 0],
        "Actual_1_Pred_1": cm[1, 1]
    })

    # ==========================================
    # Feature Importance
    # ==========================================

    importance = pd.Series(
        model.feature_importances_,
        index=features
    ).sort_values(ascending=False)

    for feature, value in importance.items():

        feature_results.append({
            "Timeframe": tf,
            "Feature": feature,
            "Importance": value
        })

    # ==========================================
    # Live Prediction
    # ==========================================

    current_market_state = test[features].tail(1)

    future_prediction = model.predict(current_market_state)

    future_probability = model.predict_proba(
        current_market_state
    )[:, 1]

    predictions_results.append({
        "Timeframe": tf,
        "Prediction": "BUY" if future_prediction[0] == 1 else "NO BUY",
        "Confidence": round(future_probability[0] * 100, 2),
        "Horizon": f"Next {lookahead} Candles"
    })

    print("\nLIVE SIGNAL")

    if future_prediction[0] == 1:
        print("BUY SIGNAL")
    else:
        print("NO BUY SIGNAL")

    print(f"Confidence: {future_probability[0] * 100:.2f}%")  


====================== TIMEFRAME: 15m ======================

[Metrics for 15m]
Accuracy: 0.7600757050881255
              precision    recall  f1-score   support

           0       0.87      0.83      0.85     48007
           1       0.39      0.48      0.43     11170

    accuracy                           0.76     59177
   macro avg       0.63      0.65      0.64     59177
weighted avg       0.78      0.76      0.77     59177


Confusion Matrix
[[39634  8373]
 [ 5825  5345]]

LIVE SIGNAL
NO BUY SIGNAL
Confidence: 28.69%

====================== TIMEFRAME: 1h =======================

[Metrics for 1h]
Accuracy: 0.774269875608437
              precision    recall  f1-score   support

           0       0.84      0.89      0.86     11960
           1       0.39      0.30      0.34      2832

    accuracy                           0.77     14792
   macro avg       0.61      0.59      0.60     14792
weighted avg       0.76      0.77      0.76     14792


Confusion Matrix
[[10597  1363]


c:\Users\admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo


[Metrics for 1d]
Accuracy: 0.8611111111111112
              precision    recall  f1-score   support

           0       0.86      1.00      0.93       527
           1       0.00      0.00      0.00        85

    accuracy                           0.86       612
   macro avg       0.43      0.50      0.46       612
weighted avg       0.74      0.86      0.80       612


Confusion Matrix
[[527   0]
 [ 85   0]]

LIVE SIGNAL
NO BUY SIGNAL
Confidence: 30.19%


In [13]:
predictions_df = pd.DataFrame(
    predictions_results
)

metrics_df = pd.DataFrame(
    metrics_results
)

feature_importance_df = pd.DataFrame(
    feature_results
)

confusion_df = pd.DataFrame(
    confusion_results
)

In [14]:
predictions_df.to_csv(

    r"C:\Users\admin\Desktop\ml_predictions.csv",

    index=False

)

metrics_df.to_csv(

    r"C:\Users\admin\Desktop\ml_metrics.csv",

    index=False

)

feature_importance_df.to_csv(

    r"C:\Users\admin\Desktop\feature_importance.csv",

    index=False

)

confusion_df.to_csv(

    r"C:\Users\admin\Desktop\confusion_matrix.csv",

    index=False

)